### 01. Descarga de datos

Esta notebook intenta generar procesos, funciones para la descarga de los productos satelitales como:
- AOD,
- Variables metereologicas
- Uso de suelo
- Elevacion
- Composicion de aerosoles
- Otras
Para luego poder ser utilizadas en la prediccion del PM2.5

In [3]:
# Library
#import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt
import numpy as np
import scipy.stats
from sklearn.metrics import mean_squared_error
from math import sqrt
import os
from os import listdir
from datetime import datetime
import pandas as pd
from matplotlib.dates import DateFormatter
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression
import requests
from pathlib import Path
import os
from dotenv import load_dotenv
import sys
from config import SITES
print("librerias ok")

librerias ok


In [4]:
#Configuracion de las variables de entorno
load_dotenv()
TOKEN_MAIAC = os.getenv("TOKEN_MAIAC")
#print(TOKEN_MAIAC)

In [17]:
# #Setear las rutas
# # Ruta raíz del proyecto
# ROOT_DIR = Path(__file__).resolve().parent.parent

# # Agregar raíz al PATH de Python
# sys.path.append(str(ROOT_DIR))

# # Carpetas del proyecto
# DATA_RAW_DIR = ROOT_DIR / "data/raw"
# MAIAC_DIR = DATA_RAW_DIR / "MAIAC"


In [5]:
# Directorio donde está el notebook
NOTEBOOK_DIR = Path.cwd()

# Subir un nivel → raíz del proyecto
ROOT_DIR = NOTEBOOK_DIR.parent.parent

# Carpetas del proyecto
DATA_RAW_DIR = ROOT_DIR / "data" / "raw"
MAIAC_DIR = DATA_RAW_DIR / "MAIAC"

print("Notebook:", NOTEBOOK_DIR)
print("Root:", ROOT_DIR)
print("MAIAC:", MAIAC_DIR)

Notebook: d:\Josefina\Proyectos\Tesis\code_py\Notebooks\00-Descarga-datos
Root: d:\Josefina\Proyectos\Tesis\code_py
MAIAC: d:\Josefina\Proyectos\Tesis\code_py\data\raw\MAIAC


In [5]:
# Configuracion del sitio
PRODUCT = "MCD19A2"
DATE = "2026-08-13"

SITES = {
    "San Pablo": {
        "west": -47.18761699953164,
        "south": -23.769825082026546,
        "east": -46.2989251168905,
        "north": -23.15380334079614
    },

    "Santiago": {
        "west": -71.0653675613302, #supizq
        "south": -33.74283779348805, #AbajDer
        "east": -70.3061109228608, #abajder
        "north": -33.07880767505593 #SupIzq
    },

"Medellin": {
        "west": -75.73349373413268,
        "south": 6.064656084590012,
        "east": -75.3546571948446,
        "north": 6.45881903338495
    },

"Mexico": {
        "west": -99.62981206846747,
        "south": 18.79096843909478,
        "east": -98.41583428026645,
        "north": 19.9977998136125
    },

}



In [ ]:
#Pr
# Configuracion
SITE = "Santiago"
OUTPUT_DIR = MAIAC_DIR
url = "https://ladsweb.modaps.eosdis.nasa.gov/api/v2/content/details"

headers = {
    "Authorization": f"Bearer {TOKEN_MAIAC}"
}

date_range = f"{DATE}..{DATE}"



BBOX = SITES[SITE]

# Convertir BBOX al formato requerido por LAADS
bbox_lads = (
    f"[BBOX]"
    f"W{BBOX['west']} "
    f"S{BBOX['south']} "
    f"E{BBOX['east']} "
    f"N{BBOX['north']}"
)

# Parámetros para la API
params = {
    "products": PRODUCT,
    "temporalRanges": date_range,
    "regions": bbox_lads,
    "formats": "json"
}


# Crear carpeta de descarga
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Buscar archivos

response = requests.get(
    url,
    headers=headers,
    params=params
)

response.raise_for_status()

data = response.json()

print(f"Archivos encontrados: {data['file_count']}")
print()


# Descargar archivos encontrados

for archivo in data["content"]:
    nombre = archivo["name"]
    download_url = archivo["downloadsLink"]
    output_file = OUTPUT_DIR / nombre

    print(f"Descargando: {nombre}")
    # Si ya existe, no lo volvemos a descargar
    if output_file.exists():
        print("  → El archivo ya existe. Se omite.")
        continue

    with requests.get(
        download_url,
        headers=headers,
        stream=True
    ) as r:

        r.raise_for_status()

        with open(output_file, "wb") as f:

            for chunk in r.iter_content(chunk_size=8192):

                if chunk:
                    f.write(chunk)

    print("  → Descarga completa")
print()
print("Proceso terminado.")

Archivos encontrados: 2

Descargando: MCD19A2.A2026225.h11v12.061.2026226170657.hdf
  → Descarga completa
Descargando: MCD19A2.A2026225.h12v12.061.2026226170110.hdf
  → Descarga completa

Proceso terminado.


In [6]:
def validate_bbox(bbox):

    if not -180 <= bbox["west"] <= 180:
        raise ValueError("La longitud west debe estar entre -180 y 180.")

    if not -180 <= bbox["east"] <= 180:
        raise ValueError("La longitud east debe estar entre -180 y 180.")

    if not -90 <= bbox["south"] <= 90:
        raise ValueError("La latitud south debe estar entre -90 y 90.")

    if not -90 <= bbox["north"] <= 90:
        raise ValueError("La latitud north debe estar entre -90 y 90.")

    if bbox["west"] >= bbox["east"]:
        raise ValueError("West debe ser menor que East.")

    if bbox["south"] >= bbox["north"]:
        raise ValueError("South debe ser menor que North.")

In [8]:
# Funcion
def aod_download (site, date):

    # Configuracion
    SITE = site
    OUTPUT_DIR = MAIAC_DIR
    url = "https://ladsweb.modaps.eosdis.nasa.gov/api/v2/content/details"
    headers = {"Authorization": f"Bearer {TOKEN_MAIAC}"}

    #Fecha de interes
    date_range = f"{date}..{date}"
    #Coordenadas
    if site not in SITES:
        raise ValueError(
            f"Sitio '{site}' no configurado. "
            f"Sitios disponibles: {list(SITES.keys())}"
        )

    BBOX = SITES[site]
    # Convertir BBOX al formato requerido por LAADS
    bbox_lads = (
        f"[BBOX]"
        f"W{BBOX['west']} "
        f"S{BBOX['south']} "
        f"E{BBOX['east']} "
        f"N{BBOX['north']}"
    )

    # Parametros para la API
    params = {
        "products": "MCD19A2",
        "temporalRanges": date_range,
        "regions": bbox_lads,
        "formats": "json"
    }

    # Crear carpeta de descarga
    #OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Buscar archivos
    response = requests.get(
        url,
        headers = headers,
        params = params
    )

    response.raise_for_status()
    data = response.json()

    # Validar disponibilidad de datos 
    if data["file_count"] == 0:
        raise RuntimeError(
            f"No se encontraron archivos de AOD "
            f"para el sitio '{site}' en la fecha {date}.")
    print(f"Archivos encontrados: {data['file_count']}")


    # Descargar archivos encontrados
    downloaded = 0
    skipped = 0
    for archivo in data["content"]:
        nombre = archivo["name"]
        download_url = archivo["downloadsLink"]
        output_file = OUTPUT_DIR / nombre
        print(f"Descargando: {nombre}")
        # Si ya existe, no lo volvemos a descargar
        if output_file.exists():
            print("El archivo ya existe. Se omite.")
            skipped += 1
            continue
        with requests.get(
            download_url,
            headers = headers,
            stream = True
        ) as r:
            
            r.raise_for_status()
            with open(output_file, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
        downloaded += 1

        print("Descarga completa")

    return {"site": site, "date": date, "files_found": data["file_count"], "files_downloaded": downloaded, "files_skipped": skipped}


In [10]:
aod_download (site = "Medellin", date = "2026-07-13")

Archivos encontrados: 1
Descargando: MCD19A2.A2026194.h10v08.061.2026198175716.hdf
El archivo ya existe. Se omite.


{'site': 'Medellin',
 'date': '2026-07-13',
 'files_found': 1,
 'files_downloaded': 0,
 'files_skipped': 1}